# PRISM Dataset EDA

**Run on:** EC2 after syncing 2-3 sessions from S3 (~60-100 GB). NOT the full 1.6 TB.

**What we already know from the data card:**
- Images are **PNG**, 2448×2048, 12-bit sensor → stored in 16-bit PNG
- Polar dirs: `0d/`, `45d/`, `90d/`, `135d/`
- Filenames: nanosecond Unix timestamps e.g. `1736200123456789012.png`
- Labels: `labels.json` at repo root, keyed `folders[session][sequence][ts]`
- Train/val split defined by dataset (train/ and val/ dirs)

**Questions this notebook answers:**
1. Actual pixel value range — max 4095 (12-bit) or 65535 (16-bit full)?
2. What do RGB vs polar images look like side by side?
3. Does DoLP visually differ between dry / wet / snow conditions?
4. Class distribution — how imbalanced are surface states and materials?
5. Per-channel Stokes statistics needed for normalization in transforms.py

In [ ]:
# ── CONFIG ───────────────────────────────────────────────────────────────────
DATA_ROOT    = "/mnt/nvme/polaris"      # local dir with train/ and val/ subdirs
LABELS_JSON  = "/mnt/nvme/polaris/labels.json"
SAMPLE_N     = 50                       # frames per session for stats (cell 8)
DISPLAY_SCALE = 4                       # downsample factor for display (2448→612)
# ─────────────────────────────────────────────────────────────────────────────

import os, sys, json, random
sys.path.insert(0, os.path.abspath(".."))

from pathlib import Path
import cv2
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
from tqdm import tqdm

from src.data.stokes import compute_stokes

plt.rcParams['figure.dpi'] = 110

root = Path(DATA_ROOT)
labels = json.load(open(LABELS_JSON))
folders = labels["folders"]

# Collect all available sessions across splits
all_sessions = []
for split in ["train", "val"]:
    split_dir = root / split
    if split_dir.exists():
        all_sessions += [(split, d.name, d) for d in sorted(split_dir.iterdir()) if d.is_dir()]

print(f"Sessions found: {len(all_sessions)}")
for split, name, _ in all_sessions:
    print(f"  [{split}] {name}")

## 1. Directory Structure — verify layout of one session

In [ ]:
_, session_name, session_dir = all_sessions[0]
print(f"Session: {session_name}")
for p in sorted(session_dir.rglob("*"))[:50]:
    rel = p.relative_to(session_dir)
    prefix = "  " * (len(rel.parts) - 1)
    print(f"{prefix}{rel.name}{'/' if p.is_dir() else ''}")

## 2. Pixel Value Range — confirm 12-bit packing

In [ ]:
_, session_name, session_dir = all_sessions[0]
seq_dir = next(d for d in sorted(session_dir.iterdir()) if d.is_dir() and d.name != "vehicle_state")

# Pick one frame stem
rgb_frames = sorted((seq_dir / "rgb").glob("*.png"))
stem = rgb_frames[10].stem
print(f"Sample frame stem: {stem}")
print(f"As seconds: {int(stem)/1e9:.3f}\n")

for angle, subdir in [("0","0d"),("45","45d"),("90","90d"),("135","135d")]:
    p = seq_dir / "polar" / subdir / f"{stem}.png"
    arr = cv2.imread(str(p), cv2.IMREAD_UNCHANGED)
    print(f"  Polar {angle:>3}° | dtype={arr.dtype} | min={arr.min():6d} | max={arr.max():6d} | shape={arr.shape}")

rgb_arr = cv2.imread(str(seq_dir / "rgb" / f"{stem}.png"), cv2.IMREAD_UNCHANGED)
print(f"  RGB          | dtype={rgb_arr.dtype} | min={rgb_arr.min():6d} | max={rgb_arr.max():6d} | shape={rgb_arr.shape}")

print()
print("12-bit packed → expect max ~4095")
print("16-bit full   → expect max ~65535")

## 3. Visual: RGB vs 4 Polar Angles — same frame

In [ ]:
BIT_DEPTH = 12   # update to 16 if cell 2 shows max ~65535
MAX_VAL = (2 ** BIT_DEPTH) - 1
sc = DISPLAY_SCALE

def load_norm(path):
    arr = cv2.imread(str(path), cv2.IMREAD_UNCHANGED).astype(np.float32)
    return np.clip(arr / MAX_VAL, 0, 1)

def load_raw(path):
    return cv2.imread(str(path), cv2.IMREAD_UNCHANGED).astype(np.float32)

rgb  = cv2.cvtColor(cv2.imread(str(seq_dir/"rgb"/f"{stem}.png"), cv2.IMREAD_UNCHANGED), cv2.COLOR_BGR2RGB)
rgb  = np.clip(rgb.astype(np.float32) / MAX_VAL, 0, 1)
p0   = load_norm(seq_dir/"polar"/"0d"/f"{stem}.png")
p45  = load_norm(seq_dir/"polar"/"45d"/f"{stem}.png")
p90  = load_norm(seq_dir/"polar"/"90d"/f"{stem}.png")
p135 = load_norm(seq_dir/"polar"/"135d"/f"{stem}.png")

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
for ax, title, img in zip(axes.flat,
    ["RGB", "Polar 0°", "Polar 45°", "Polar 90°", "Polar 135°", ""],
    [rgb, p0, p45, p90, p135, None]):
    if img is None: ax.axis("off"); continue
    disp = img[::sc, ::sc] if img.ndim == 2 else img[::sc, ::sc, :]
    cmap = None if img.ndim == 3 else "gray"
    ax.imshow(disp, cmap=cmap, vmin=0, vmax=1)
    ax.set_title(title, fontsize=13); ax.axis("off")

plt.suptitle(f"{session_name} / {seq_dir.name} / frame {stem[:20]}...", fontsize=10)
plt.tight_layout(); plt.show()

## 4. Stokes Parameters and DoLP Heatmap

In [ ]:
i0   = load_raw(seq_dir/"polar"/"0d"/f"{stem}.png")
i45  = load_raw(seq_dir/"polar"/"45d"/f"{stem}.png")
i90  = load_raw(seq_dir/"polar"/"90d"/f"{stem}.png")
i135 = load_raw(seq_dir/"polar"/"135d"/f"{stem}.png")

stokes = compute_stokes(i0, i45, i90, i135, eps=1.0)

print("Stokes channel statistics:")
for name, arr in stokes.items():
    print(f"  {name:5s} | min={arr.min():.4f}  max={arr.max():.4f}  mean={arr.mean():.4f}  std={arr.std():.4f}")

fig, axes = plt.subplots(1, 5, figsize=(22, 5))
cmaps = {"S0":"gray", "S1":"RdBu_r", "S2":"RdBu_r", "DoLP":"hot", "AoLP":"hsv"}
for ax, (name, arr) in zip(axes, stokes.items()):
    im = ax.imshow(arr[::sc, ::sc], cmap=cmaps[name])
    ax.set_title(name, fontsize=13); ax.axis("off")
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

plt.suptitle("High DoLP on road = wet/icy surface (Fresnel reflection)", fontsize=11)
plt.tight_layout(); plt.show()

## 5. Label Lookup — verify for sample frames

In [ ]:
from src.data.utils import lookup_label, _parse_frame_ts as _ns_to_sec

_, session_name, session_dir = all_sessions[0]
folder_entry = folders.get(session_name, {})

print(f"Session: {session_name}")
print(f"Label structure keys: {list(folder_entry.keys())}\n")

# Test lookup on 5 frames
seq_dir = next(d for d in sorted(session_dir.iterdir()) if d.is_dir() and d.name != "vehicle_state")
frames = sorted((seq_dir / "rgb").glob("*.png"))[:5]

print(f"{'Stem (ns)':>25}  {'Seconds':>16}  Label")
print("-" * 75)
for f in frames:
    stem = f.stem
    ts_sec = _ns_to_sec(stem)
    label = lookup_label(folder_entry, ts_sec)
    print(f"  {stem:>25}  {ts_sec:>16.3f}  {label}")

## 6. Class Distribution Across All Available Sessions

In [ ]:
records = []

for split, session_name, session_dir in tqdm(all_sessions, desc="indexing"):
    folder_entry = folders.get(session_name)
    if not folder_entry:
        continue
    for seq_dir in sorted(session_dir.iterdir()):
        if not seq_dir.is_dir() or seq_dir.name == "vehicle_state":
            continue
        rgb_dir = seq_dir / "rgb"
        if not rgb_dir.exists():
            continue
        for f in rgb_dir.glob("*.png"):
            ts_sec = _ns_to_sec(f.stem)
            label = lookup_label(folder_entry, ts_sec)
            if label:
                records.append({
                    "split":            split,
                    "session":          session_name,
                    "surface_state":    label.get("surface_state"),
                    "surface_material": label.get("surface_material"),
                    "weather":          label.get("weather"),
                    "road_type":        label.get("road_type"),
                })

df = pd.DataFrame(records)
print(f"Total labeled frames: {len(df)}")
print(f"Excluded (transitions): {sum(1 for _ in records) - len(df)}")
print(df.head())

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

for ax, col in zip(axes.flat, ["surface_state", "surface_material", "weather", "road_type"]):
    counts = df[col].value_counts()
    counts.plot.bar(ax=ax, color="steelblue", edgecolor="white")
    ax.set_title(col.replace("_", " ").title(), fontsize=13)
    ax.set_xlabel("")
    for bar, count in zip(ax.patches, counts):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 50,
                f"{count:,}", ha='center', va='bottom', fontsize=9)

plt.suptitle("PRISM Class Distribution (sampled sessions)", fontsize=13)
plt.tight_layout(); plt.show()

print("\nSurface state distribution:")
print(df["surface_state"].value_counts().to_string())
print("\nSurface material distribution:")
print(df["surface_material"].value_counts().to_string())

## 7. DoLP Comparison Across Conditions

In [ ]:
# Pick one session per available condition for visual comparison
# Assign sessions manually here after seeing cell 6 distribution

CONDITION_MAP = {}  # e.g. {"dry": all_sessions[0], "wet": all_sessions[1]}
for split, name, sdir in all_sessions:
    fe = folders.get(name, {})
    segs = fe.get("ordered_segments", [])
    default = fe.get("default_label", {})
    state = (segs[0]["label"].get("surface_state") if segs
             else default.get("surface_state", "unknown"))
    if state not in CONDITION_MAP:
        CONDITION_MAP[state] = (split, name, sdir)

print("Condition → session mapping:")
for cond, (split, name, _) in CONDITION_MAP.items():
    print(f"  {cond:15s} → [{split}] {name}")

fig, axes = plt.subplots(len(CONDITION_MAP), 2, figsize=(12, 4 * len(CONDITION_MAP)))
if len(CONDITION_MAP) == 1: axes = [axes]

for row, (cond, (split, name, sdir)) in enumerate(CONDITION_MAP.items()):
    seq = next(d for d in sorted(sdir.iterdir()) if d.is_dir() and d.name != "vehicle_state")
    s = sorted((seq/"rgb").glob("*.png"))[5].stem

    rgb_img = cv2.cvtColor(cv2.imread(str(seq/"rgb"/f"{s}.png"), cv2.IMREAD_UNCHANGED), cv2.COLOR_BGR2RGB)
    rgb_img = np.clip(rgb_img.astype(np.float32) / MAX_VAL, 0, 1)

    i0_  = load_raw(seq/"polar"/"0d"/f"{s}.png")
    i45_ = load_raw(seq/"polar"/"45d"/f"{s}.png")
    i90_ = load_raw(seq/"polar"/"90d"/f"{s}.png")
    i135_= load_raw(seq/"polar"/"135d"/f"{s}.png")
    st   = compute_stokes(i0_, i45_, i90_, i135_)

    axes[row][0].imshow(rgb_img[::sc,::sc,:])
    axes[row][0].set_title(f"{cond.upper()} — RGB"); axes[row][0].axis("off")
    im = axes[row][1].imshow(st["DoLP"][::sc,::sc], cmap="hot", vmin=0, vmax=0.5)
    axes[row][1].set_title(f"{cond.upper()} — DoLP (mean={st['DoLP'].mean():.3f})")
    axes[row][1].axis("off"); plt.colorbar(im, ax=axes[row][1])

plt.tight_layout(); plt.show()

## 8. Per-Channel Stokes Statistics (for transforms.py normalization)

In [ ]:
channel_names = ["S0", "S1", "S2", "DoLP", "AoLP"]
all_means = {c: [] for c in channel_names}
all_stds  = {c: [] for c in channel_names}

for split, session_name, session_dir in tqdm(all_sessions):
    for seq_dir in sorted(session_dir.iterdir()):
        if not seq_dir.is_dir() or seq_dir.name == "vehicle_state":
            continue
        frames = sorted((seq_dir/"polar"/"0d").glob("*.png"))
        sampled = random.sample(frames, min(SAMPLE_N, len(frames)))
        for f in sampled:
            stem = f.stem
            try:
                i0_  = load_raw(seq_dir/"polar"/"0d"/f"{stem}.png")
                i45_ = load_raw(seq_dir/"polar"/"45d"/f"{stem}.png")
                i90_ = load_raw(seq_dir/"polar"/"90d"/f"{stem}.png")
                i135_= load_raw(seq_dir/"polar"/"135d"/f"{stem}.png")
                st = compute_stokes(i0_, i45_, i90_, i135_)
                for ch in channel_names:
                    all_means[ch].append(float(st[ch].mean()))
                    all_stds[ch].append(float(st[ch].std()))
            except Exception as e:
                pass

print("\nPolar channel statistics — copy these into src/data/transforms.py:\n")
polar_mean, polar_std = [], []
for ch in channel_names:
    m = float(np.mean(all_means[ch]))
    s = float(np.mean(all_stds[ch]))
    polar_mean.append(round(m, 6))
    polar_std.append(round(s, 6))
    print(f"  {ch:5s}  mean={m:.4f}  std={s:.4f}")

print(f"\npolar_mean = {polar_mean}")
print(f"polar_std  = {polar_std}")

## Summary — What to Update Before Training

| Finding | File to update |
|---|---|
| Pixel max value (4095 or 65535) | [src/data/dataset.py](../src/data/dataset.py) — `_load_rgb()` division constant |
| `polar_mean` / `polar_std` values | [src/data/transforms.py](../src/data/transforms.py) |
| Class imbalance (rare: snow, slush, gravel) | [src/training/trainer.py](../src/training/trainer.py) — add `class_weight` to CrossEntropyLoss |